# Detección de anomalías para identificación de fraude en comercio electrónico

* El objetivo de este trabajo es poder abordar uno de los casos de uso más habituales en la detección de anomalías: la identificación de fraude.
* Para ello, nos apoyaremos en un conjunto de datos sintético sobre transacciones financieras, que simula la complejidad y riqueza de estos datos (https://www.kaggle.com/datasets/shriyashjagtap/fraudulent-e-commerce-transactions/data)
* En este contexto, el fraude hace referencia a causísticas como el uso de datos de tarjeta de crédito o débito obtenidos ilegalmente (por phishing, malware o filtraciones de datos) para realizar una compra en línea.
* El enfoque de detección de anomalías basado en el uso de soluciones no supervisadas es clave debido a que las tácticas fraudulentas evolucionan constantemente.
* Por ello, trabajaremos distintos tipos de enfoques de detección de anomalías en este contexto: puntuales, contextuales y colectivas, de cara a ver qué enfoque es el más efectivo.


### Descripción de Campos de Transacción

---

#### Identificadores Únicos

* **Transaction ID (ID de Transacción):** Un identificador único para cada transacción.
* **Customer ID (ID de Cliente):** Un identificador único para cada cliente.

---

#### Detalles Financieros y Temporales

* **Transaction Amount (Monto de la Transacción):** La cantidad total de dinero intercambiada en la transacción.
* **Transaction Date (Fecha de Transacción):** La fecha y hora en que tuvo lugar la transacción.
* **Transaction Hour (Hora de la Transacción):** La hora del día en que ocurrió la transacción.

---

#### Detalles del Producto y Pago

* **Payment Method (Método de Pago):** El método utilizado para completar la transacción (por ejemplo, tarjeta de crédito, PayPal, etc.).
* **Product Category (Categoría de Producto):** La categoría del producto involucrado en la transacción.
* **Quantity (Cantidad):** El número de productos involucrados en la transacción.

---

#### Información del Cliente y Envío

* **Customer Age (Edad del Cliente):** La edad del cliente que realiza la transacción.
* **Customer Location (Ubicación del Cliente):** La ubicación geográfica del cliente.
* **Shipping Address (Dirección de Envío):** La dirección donde se envió el producto.
* **Billing Address (Dirección de Facturación):** La dirección asociada con el método de pago.
* **Account Age Days (Antigüedad de la Cuenta en Días):** La antigüedad de la cuenta del cliente en días en el momento de la transacción.

---

#### Detalles Técnicos y Detección de Fraude

* **Device Used (Dispositivo Utilizado):** El tipo de dispositivo utilizado para realizar la transacción (por ejemplo, móvil, escritorio).
* **IP Address (Dirección IP):** La dirección IP del dispositivo utilizado para la transacción.
* **Is Fraudulent (Es Fraudulenta):** Un indicador binario de si la transacción es fraudulenta (1 para fraudulenta, 0 para legítima).

## 0. Carga de Librerías y Definir funciones comunes

In [1]:
# Instalar dependencias necesarias
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
        print(f"{package} ya está instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Instalar ruptures si no está disponible
install_package("ruptures")

Instalando ruptures...


In [2]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import warnings
import os

# Configurar warnings y estilo de visualización
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

# Configurar semilla para reproducibilidad
np.random.seed(42)

print("Librerías importadas correctamente")

Librerías importadas correctamente


### 0.1. Funciones comunes

In [ ]:
# Aqui puedes incluir tus funciones comunes para el código

## 1. Cargar datos

In [3]:
# Cargar los datasets
df_train = pd.read_csv('Fraudulent_E-Commerce_Transaction_Data.csv', on_bad_lines='skip').sample(frac=0.1).reset_index(drop=True)
df_test = pd.read_csv('Fraudulent_E-Commerce_Transaction_Data_2.csv', on_bad_lines='skip').sample(frac=0.2).reset_index(drop=True)

print(f'\nDimensiones del dataset de entrenamiento: {df_train.shape}')
print(f'Dimensiones del dataset de prueba: {df_test.shape}')

# Mostrar las primeras filas del dataset de entrenamiento
print('\nPrimeras 5 filas del dataset de entrenamiento:')
display(df_train.head())


Dimensiones del dataset de entrenamiento: (147295, 16)
Dimensiones del dataset de prueba: (4727, 16)

Primeras 5 filas del dataset de entrenamiento:


,Transaction ID,Customer ID,Transaction Amount,Transaction Date,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,Is Fraudulent,Account Age Days,Transaction Hour
0,69c99073-bf9a-4f39-87d6-0bac726fe43b,37fc6628-9d40-477c-b576-4e739af7eb51,84.91,2024-03-30 19:07:37,bank transfer,electronics,1,31,South Richard,tablet,78.16.79.0,297 Anthony Creek Suite 388\nNorth Michaelfurt...,297 Anthony Creek Suite 388\nNorth Michaelfurt...,0,34,19
1,7eae0592-d64b-45f0-aa01-b4d224328a0b,14f1b38e-72f1-4a82-aa5a-d422aa990279,30.99,2024-02-23 02:10:27,credit card,electronics,3,47,Amberside,mobile,102.33.181.96,USNV Diaz\nFPO AA 86306,USNV Diaz\nFPO AA 86306,0,95,2
2,16ad4695-16d5-4b6e-b65d-0308473d82c1,a7f6f1a6-220d-44eb-b8b8-e27a655e5cdc,310.32,2024-01-05 19:14:48,debit card,clothing,4,50,South Brianburgh,mobile,139.222.215.225,"588 Darrell Neck\nTannerville, OR 16918","588 Darrell Neck\nTannerville, OR 16918",0,304,19
3,271e4574-c2cc-422e-96cd-2ac4cb25d8e0,801e4f9f-ed11-4992-9ff1-d2ab329e72b7,40.50,2024-02-18 15:44:20,bank transfer,home & garden,1,38,Christianfort,tablet,166.120.49.97,"98931 Teresa Glen\nEast Donna, AZ 79166","98931 Teresa Glen\nEast Donna, AZ 79166",0,14,15
4,e409f671-5c17-43f0-a7ed-06d2a648c5eb,0f97f0b6-0575-4e51-8b5d-571e2ec27cd3,537.91,2024-02-25 13:48:44,debit card,home & garden,2,31,Sarafort,tablet,144.152.197.120,"11610 Miller Ford Suite 999\nTaraborough, MN 2...","11610 Miller Ford Suite 999\nTaraborough, MN 2...",0,274,13


In [7]:
print('\nPrimeras 5 filas del dataset de test:')
display(df_test.head())


Primeras 5 filas del dataset de test:


,Transaction ID,Customer ID,Transaction Amount,Transaction Date,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,Is Fraudulent,Account Age Days,Transaction Hour
0,e231961e-b47d-4acb-ac03-1f6e0736b36c,35ddf9e3-40b1-48ae-b4b9-e8a37e9d8963,71.84,2024-03-05 04:22:29,credit card,health & beauty,1,39,Westview,mobile,11.180.184.121,"912 Wright Pass\nEast Brandonmouth, NJ 86548","912 Wright Pass\nEast Brandonmouth, NJ 86548",0,204,4
1,b98d1ae3-8cf5-45a6-8747-b9273ea0f5df,1ff29849-e275-4f9b-94f5-3ebecab008b3,252.40,2024-01-24 20:30:47,debit card,clothing,4,26,Lake Davidstad,desktop,213.163.25.76,"815 Davis Village Suite 432\nSouth Scott, MA 8...","91624 Pugh Locks Apt. 350\nWest Richard, KY 69308",0,320,20
2,328921c6-ccb9-4271-b950-a08b7208a89a,e39ee88c-3ddd-4b34-8fb8-81647875fb7c,878.39,2024-02-06 14:14:25,bank transfer,home & garden,1,27,Paulville,mobile,185.253.30.101,"247 Robles Stream\nLake Joanna, VT 23235","247 Robles Stream\nLake Joanna, VT 23235",0,224,14
3,8ecb08ec-e88e-48a3-84bd-781be224fff8,9f30889b-c919-4592-a0d4-15310ce370da,199.00,2024-01-21 20:57:33,debit card,electronics,5,41,West Robert,mobile,157.68.145.188,"154 Jacob Point Suite 485\nLake Robert, TX 04985","154 Jacob Point Suite 485\nLake Robert, TX 04985",0,303,20
4,71b3c292-5aa2-4718-a7dc-2266210f806e,f79d630f-cfe7-46d3-a6e2-f5ed295ecd62,197.87,2024-01-26 13:25:09,debit card,electronics,5,38,North Zachary,desktop,153.60.20.241,"81304 Lindsey Freeway\nEdwardsborough, HI 10041","81304 Lindsey Freeway\nEdwardsborough, HI 10041",0,51,13


## 2. Análisis y procesamiento inicial

#### PREGUNTA 1:
* Realiza un análisis exploratorio de datos (EDA). ¿Qué insights se derivan de los datos disponibles?

#### RESPUESTA:

#### PREGUNTA 2:
* Aplica los preprocesamientos iniciales en los datos

#### RESPUESTA:


## 3. Detección de anomalías puntuales


#### PREGUNTA 3:
* Considerando 'Transaction Amount' como variable para tratar de predecir el fraude desde ella, aplica métodos estadísticos univariantes para detección de anomalías.
* Analiza los resultados obtenidos.


#### RESPUESTA:

####

## 4. Detección de anomalías contextuales


#### PREGUNTA 4:
* Considera métodos de detección de anomalías que permitan ver anomalías contextuales de manera no supervisada, y aplícalos sobre los datos.
* Analiza los resultados obtenidos.

#### RESPUESTA:

### 5. Detección de anomalías colectivas
* Sliding window
* PELT

#### PREGUNTA 5:
* Considera ahora métodos de detección de anomalías que permitan ver anomalías colectivas de manera univariante, sobre la variable 'Transaction Amount'. Aplica estos métodos.
* Analiza los resultados obtenidos.

#### RESPUESTA: